In [ ]:
import os
import shutil

SOURCE_DIR = "pynguin_tests"
DEST_DIR = "Generated_Tests"

# Create the destination directory if it doesn't exist
os.makedirs(DEST_DIR, exist_ok=True)

counter = 0
for root, _, files in os.walk(SOURCE_DIR):
    for file in files:
        if file.endswith(".py"):
            source_path = os.path.join(root, file)
            dest_path = os.path.join(DEST_DIR, file)

            # If file already exists at destination, optionally skip or overwrite
            if os.path.exists(dest_path):
                print(f"⚠️ Skipped (already exists): {dest_path}")
                continue

            shutil.copy2(source_path, dest_path)
            print(f"✅ Copied: {source_path} → {dest_path}")
            counter += 1

print(f"\n🎉 Done! Total .py files copied: {counter}")


In [14]:
import os
import subprocess
import csv
import json

# Directory where both test and code files are now located
TEST_AND_CODE_DIR = "Generated_Tests"
CODE_DIR = "code_files_600"
CSV_FILE = "coverage_results.csv"

# Set PYTHONPATH so that imports like "import code_snippet_0" work
os.environ["PYTHONPATH"] = os.path.abspath(CODE_DIR)

# CSV Headers
headers = ["Test_File", "Code_File", "Line_Coverage(%)", "Branch_Coverage(%)"]
results = []

# Collect existing test files
existing_test_files = sorted([f for f in os.listdir(TEST_AND_CODE_DIR) if f.startswith("test_code_snippet_") and f.endswith(".py")])

# Loop through test files
for test_file in existing_test_files:
    test_path = os.path.join(TEST_AND_CODE_DIR, test_file)

    try:
        subprocess.run(["coverage", "erase"], check=True)
        subprocess.run(["coverage", "run", "--branch", "-m", "pytest", test_path], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        subprocess.run(["coverage", "json", "-o", "cov.json"], check=True)

        with open("cov.json") as f:
            data = json.load(f)

        totals = data.get("totals", {})
        line_coverage = totals.get("percent_covered", 0)
        branch_coverage = totals.get("percent_branches_covered", 0)

        matching_code_file = test_file.replace("test_", "").replace(".py", "") + ".py"
        results.append([test_file, matching_code_file, line_coverage, branch_coverage])
        print(f"✅ Processed: {test_file} → {matching_code_file}")

    except subprocess.CalledProcessError:
        print(f"❌ Error running: {test_file}")
        results.append([test_file, "?", 0, 0])

# Add untested code files (those with no matching test file)
all_code_files = {f"code_snippet_{i}.py" for i in range(600)}
tested_code_files = {row[1] for row in results}

untested_code_files = all_code_files - tested_code_files

for code_file in sorted(untested_code_files):
    test_file = code_file.replace("code_", "test_code_")
    results.append([test_file, code_file, "No Test", "No Test"])
    print(f"⚠️ No test found for: {code_file}")

# Save to CSV
with open(CSV_FILE, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(headers)
    writer.writerows(results)

print(f"\n📄 Coverage report saved to: {CSV_FILE}")


✅ Processed: test_code_snippet_0.py → code_snippet_0.py
❌ Error running: test_code_snippet_1.py
❌ Error running: test_code_snippet_10.py
❌ Error running: test_code_snippet_100.py
❌ Error running: test_code_snippet_101.py
✅ Processed: test_code_snippet_104.py → code_snippet_104.py
✅ Processed: test_code_snippet_107.py → code_snippet_107.py
✅ Processed: test_code_snippet_108.py → code_snippet_108.py
✅ Processed: test_code_snippet_112.py → code_snippet_112.py
✅ Processed: test_code_snippet_113.py → code_snippet_113.py
✅ Processed: test_code_snippet_114.py → code_snippet_114.py
❌ Error running: test_code_snippet_116.py
✅ Processed: test_code_snippet_117.py → code_snippet_117.py
❌ Error running: test_code_snippet_119.py
❌ Error running: test_code_snippet_120.py
✅ Processed: test_code_snippet_125.py → code_snippet_125.py
❌ Error running: test_code_snippet_126.py
❌ Error running: test_code_snippet_127.py
❌ Error running: test_code_snippet_128.py
✅ Processed: test_code_snippet_13.py → code_sni